# HAllA from scratch: Docker, a real paired dataset, and a customised hallagram

This notebook runs [HAllA](https://github.com/biobakery/halla) end to end without installing it
on your machine. HAllA lives inside a Docker image; this notebook's Python kernel calls that
image, then reads the results back and draws the figures itself.

Work through [`../README.md`](../README.md) alongside it.

**Before you start:** Docker Desktop must be running, and the image must be built (Section 2
below will build it for you the first time, which takes a few minutes).

## 1. Setup

In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline


def find_repo_root(start=None):
    '''Walk up from the notebook until we find the repository root.'''
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "hallagram_render.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find hallagram_render.py above this notebook. "
        "Run the notebook from inside a clone of the repository."
    )


REPO = find_repo_root()
DATA = REPO / "data"
RUNS = REPO / "halla_runs"       # HAllA output, one subdirectory per run (git-ignored)
FIGS = REPO / "figures"
RUNS.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

# The renderer is a plain module in the repository root. Reloading it means you can edit
# hallagram_render.py and re-run the cells below without restarting the kernel.
sys.path.insert(0, str(REPO))
import hallagram_render

importlib.reload(hallagram_render)
build_hallagram_inputs = hallagram_render.build_hallagram_inputs
render_native_hallagram = hallagram_render.render_native_hallagram

print("repository:", REPO)

## 2. The input files

HAllA takes two tab-separated matrices with **features as rows and samples as columns**, and the
first header cell must be the literal string `#`.

Both matrices must also carry the *same samples in the same order*.

In [ ]:
X = pd.read_csv(DATA / "X_metabolites.txt", sep="\t", index_col=0)   # 40 metabolites
Y = pd.read_csv(DATA / "Y_taxa.txt", sep="\t", index_col=0)          # 30 genera

print(f"X: {X.shape[0]} metabolites x {X.shape[1]} samples")
print(f"Y: {Y.shape[0]} genera      x {Y.shape[1]} samples")
assert list(X.columns) == list(Y.columns), "X and Y must carry the same samples in the same order"

display(X.iloc[:3, :4])
display(Y.iloc[:3, :4])

Y holds `log10(relative abundance + 1e-5)`, so a genus absent from a sample sits at the floor
value of `-5`. X holds the metabolite values as the source collection published them, already
log-scaled. Neither needs further transformation here.

## 3. Build the Docker image

This checks whether the image exists and builds it if not. The build installs Python 3.9, R 4.3,
two R packages and HAllA itself, so the first run takes a few minutes and about 6 GB of disk.
Afterwards it is instant.

In [ ]:
IMAGE = "halla-tutorial:0.8.40"
BUILD_IF_MISSING = True


def image_exists(tag):
    result = subprocess.run(["docker", "images", "-q", tag], capture_output=True, text=True)
    return bool(result.stdout.strip())


if image_exists(IMAGE):
    print(f"image present: {IMAGE}")
elif BUILD_IF_MISSING:
    print(f"building {IMAGE} — this takes a few minutes...")
    build = subprocess.run(["docker", "build", "-t", IMAGE, str(REPO / "docker")],
                           capture_output=True, text=True)
    if build.returncode != 0:
        print(build.stdout[-2000:])
        print(build.stderr[-2000:])
        raise RuntimeError("docker build failed. Is Docker Desktop running?")
    print(f"built {IMAGE}")
else:
    raise RuntimeError(f"{IMAGE} not found. Build it with:  docker build -t {IMAGE} docker/")

## 4. Run HAllA

Four runs of the same data with different settings. The first is the one to read as "the result";
the other three exist so Section 8 can show you what the settings actually do.

Each run mounts the repository into the container at `/halla` and calls the `halla` CLI directly.
Nothing is installed locally, and all paths passed to HAllA are relative to the mount point.

In [ ]:
RUN_CONFIGS = {
    "spearman_average": dict(
        metric="spearman", linkage_method="average", fdr_alpha=0.05,
        label="Spearman, average linkage (the sensible default)",
    ),
    "pearson_complete_signed": dict(
        metric="pearson", linkage_method="complete", fdr_alpha=0.05,
        sim2dist_disable_abs=True,
        label="Pearson, complete linkage, signed distance",
    ),
    "spearman_loose_fdr": dict(
        metric="spearman", linkage_method="average", fdr_alpha=0.20,
        label="Spearman, average linkage, FDR relaxed to 0.20",
    ),
    "nmi_average": dict(
        metric="nmi", linkage_method="average", fdr_alpha=0.05,
        label="Normalised mutual information, average linkage",
    ),
}

FORCE_RERUN = False   # set True to re-run even where output already exists


def build_halla_cmd(name, cfg):
    '''Assemble the full `docker run ... halla ...` command for one configuration.'''
    cmd = [
        "docker", "run", "--rm",
        "-v", f"{REPO}:/halla",     # mount the repository into the container
        "-w", "/halla",             # ...and start there, so relative paths work
        IMAGE,
        "halla",
        "-x", "data/X_metabolites.txt",
        "-y", "data/Y_taxa.txt",
        "-m", cfg["metric"],                            # -m, not --metric
        "-o", f"halla_runs/{name}",
        "--fdr_alpha", str(cfg["fdr_alpha"]),
        "--fnr_thresh", str(cfg.get("fnr_thresh", 0.2)),
        "--linkage_method", cfg["linkage_method"],
        "--hallagram", "--plot_file_type", "png",       # HAllA's own hallagram; pdf for vector output
        "--seed", "42",
    ]
    if cfg.get("sim2dist_disable_abs"):
        cmd.append("--sim2dist_disable_abs")            # a bare flag, it takes no value
    return cmd


for name, cfg in RUN_CONFIGS.items():
    out_dir = RUNS / name
    if (out_dir / "all_associations.txt").exists() and not FORCE_RERUN:
        print(f"[{name}] already done, skipping (set FORCE_RERUN = True to redo)")
        continue
    # HAllA refuses to write into a directory that already exists, and it cannot create
    # a nested path, so clear the run directory and make sure its parent is there.
    if out_dir.exists():
        shutil.rmtree(out_dir)
    RUNS.mkdir(exist_ok=True)
    print(f"[{name}] running...")
    result = subprocess.run(build_halla_cmd(name, cfg), capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
        raise RuntimeError(f"HAllA failed for '{name}'")
    print(f"[{name}] done")

## 5. What HAllA wrote

`all_associations.txt` holds **every** feature pair with its similarity score and q-value.

`X.tsv` and `Y.tsv` are HAllA's own copies of the inputs. The renderer in Section 6 reads them to
recover feature order, which is why you should never pass `--dont_copy`.

In [ ]:
run_dir = RUNS / "spearman_average"
print(sorted(p.name for p in run_dir.iterdir()))

assoc = pd.read_csv(run_dir / "all_associations.txt", sep="\t")
print(f"\nall_associations.txt: {len(assoc)} rows = every one of the "
      f"{X.shape[0]} x {Y.shape[0]} pairs")
display(assoc.sort_values("q-values").head(5))

clusters = pd.read_csv(run_dir / "sig_clusters.txt", sep="\t")
sizes = pd.DataFrame({
    "rank": clusters["cluster_rank"],
    "n_metabolites": clusters["cluster_X"].str.count(";") + 1,
    "n_taxa": clusters["cluster_Y"].str.count(";") + 1,
    "best_q": clusters["best_adjusted_pvalue"],
})
print(f"\nsig_clusters.txt: {len(clusters)} blocks; the largest few:")
display(sizes.head(6))

## 6. A customised hallagram

`--hallagram` already gave you `hallagram.png` in the run directory. The renderer in
`hallagram_render.py` redraws the same result in your own kernel so you can annotate it:
categorical and numeric metadata strips on either axis, readable tick labels, and (Section 7) a
marginal barplot.

`build_hallagram_inputs` reads the run directory and returns the similarity matrix, the
FDR-rejection mask, the significant blocks and the two dendrogram orderings.

The metadata frames are indexed by feature ID, and those IDs must match the row names of
`X_metabolites.txt` and `Y_taxa.txt` exactly. Anything that does not match raises an error.

In [ ]:
# Read metadata with an explicit sep="\t". Do not let pandas sniff the delimiter here:
# several metabolite names contain commas, and the sniffer will guess CSV and mangle them.
x_meta_raw = pd.read_csv(DATA / "metabolite_metadata.txt", sep="\t", index_col=0)
y_meta_raw = pd.read_csv(DATA / "taxa_metadata.txt", sep="\t", index_col=0)
x_meta_raw.index = x_meta_raw.index.map(str)
y_meta_raw.index = y_meta_raw.index.map(str)

SHANNON_STRIP = "Shannon corr."
SHANNON_ALPHA = 0.05


def prepare_x_metadata(df):
    '''Derive the numeric strip: the Shannon correlation, blanked where it fails FDR.

    Deriving it here rather than in the data file means you can change SHANNON_ALPHA and
    re-render without touching prepare_data.py. NaN cells render in the missing-value grey.
    '''
    df = df.copy()
    df[SHANNON_STRIP] = df["shannon_rho"].where(df["shannon_fdr"] < SHANNON_ALPHA)
    return df


METRIC_LABELS = {
    "spearman": "Spearman rho", "pearson": "Pearson r", "nmi": "NMI",
    "mi": "MI", "dcor": "Distance correlation", "xicor": "xi correlation",
}

SETTINGS = dict(
    dpi=200,
    cmap="BrBG_r",              # the association scale; deliberately not the strip colormap
    trim=True,                  # drop features that appear in no displayed block
    block_num=15,               # how many top blocks to outline; -1 for all
    x_metadata_columns=["Superpathway", SHANNON_STRIP],
    y_metadata_columns=["Phylum"],
    x_tick_label_col="label",   # show the readable name, not the sanitised feature ID
    y_tick_label_col="label",
    numeric_cmaps={SHANNON_STRIP: "PuOr_r"},
    numeric_center_zero=[SHANNON_STRIP],
    na_legend_labels={SHANNON_STRIP: "Not significant"},
    hide_na_legend=[SHANNON_STRIP],
    legend_titles={
        "Superpathway": "Metabolite superpathway",
        SHANNON_STRIP: "Correlation with\nShannon diversity\n(FDR < 0.05)",
        "Phylum": "Bacterial phylum",
    },
    x_label_size=8, y_label_size=9, metadata_font_size=9,
    legend_scale=1.0,
    # Pin the categorical colours. Without this the palette is assigned from whichever
    # categories survive trimming, so the same phylum gets a different colour in each
    # figure and the comparison in Section 8 becomes hard to read.
    category_orders={
        "Superpathway": ["Amino Acid", "Peptide", "Carbohydrate", "Lipid", "Nucleotide",
                         "Cofactors and Vitamins", "Energy", "Xenobiotics"],
        "Phylum": ["Actinobacteriota", "Bacteroidota", "Firmicutes", "Firmicutes_A"],
    },
    category_colors={
        "Superpathway": {
            "Amino Acid": "#4C72B0", "Peptide": "#A6CEE3", "Carbohydrate": "#55A868",
            "Lipid": "#C44E52", "Nucleotide": "#8172B2", "Cofactors and Vitamins": "#CCB974",
            "Energy": "#64B5CD", "Xenobiotics": "#DD8452",
        },
        "Phylum": {
            "Actinobacteriota": "#B15928", "Bacteroidota": "#1F78B4",
            "Firmicutes": "#E31A1C", "Firmicutes_A": "#FDBF6F",
        },
    },
)


def render(name, y_barplot=None, **overrides):
    '''Render one run. Keyword arguments override SETTINGS for this call only.'''
    cfg = RUN_CONFIGS[name]
    settings = {**SETTINGS, **overrides}
    settings.setdefault("cbar_label", METRIC_LABELS.get(cfg["metric"], "Association"))
    inputs = build_hallagram_inputs(str(RUNS / name))
    fig, _ = render_native_hallagram(
        inputs,
        x_metadata_df=prepare_x_metadata(x_meta_raw),
        y_metadata_df=y_meta_raw,
        y_barplot=y_barplot,
        **settings,
    )
    return fig


inputs = build_hallagram_inputs(str(RUNS / "spearman_average"))
print(f"{len(inputs['x_features'])} metabolites x {len(inputs['y_features'])} genera, "
      f"{len(inputs['significant_blocks'])} significant blocks")

fig = render("spearman_average")
fig.savefig(FIGS / "hallagram_custom.png", dpi=200, bbox_inches="tight")
display(fig)
plt.close(fig)

## 7. Adding a marginal barplot

`y_barplot` draws a horizontal bar per Y feature in the left margin, aligned to the heatmap rows.
Anything you can compute per feature works — here it is prevalence, the percentage of samples in
which the genus was detected at all.

Prevalence is computed straight from the abundance table, so it does not depend on HAllA.

In [ ]:
prevalence = pd.read_csv(DATA / "taxa_prevalence.txt", sep="\t")
prev_map = dict(zip(prevalence["feature"].astype(str), prevalence["prevalence_pct"].astype(float)))
print(f"prevalence spans {min(prev_map.values()):.0f}% to {max(prev_map.values()):.0f}% "
      f"of {int(prevalence['n_samples'].iloc[0])} samples")

# The same numbers, recomputed from Y to show where they come from:
recomputed = ((10 ** Y > 1e-5).mean(axis=1) * 100).round(2)
print("matches the committed file:",
      bool(np.allclose(recomputed.reindex(prevalence['feature']).values,
                       prevalence["prevalence_pct"].values, atol=0.01)))

fig = render(
    "spearman_average",
    y_barplot={"values": prev_map, "label": "Prevalence %", "vmax": 100},
)
fig.savefig(FIGS / "hallagram_prevalence.png", dpi=200, bbox_inches="tight")
display(fig)
plt.close(fig)

## 8. What the settings actually do

Same data, four sets of settings. `performance.txt` records the configuration and the result
counts for each run, which makes it the fastest way to compare them.

In [ ]:
WANTED = {
    "pdist metric": "metric",
    "linkage method": "linkage",
    "sim2dist set abs": "abs. distance",
    "fdr alpha": "FDR alpha",
    "Number of significant associations": "sig. pairs",
    "Number of significant clusters": "sig. blocks",
    "Total execution time": "runtime",
}


def parse_performance(path):
    values = {}
    for line in Path(path).read_text().splitlines():
        key, sep, value = line.partition(":")
        if sep and key.strip() in WANTED:
            values[WANTED[key.strip()]] = value.strip()
    return values


summary = pd.DataFrame(
    {name: parse_performance(RUNS / name / "performance.txt") for name in RUN_CONFIGS}
).T[list(WANTED.values())]
summary.insert(0, "largest block", [
    "{}x{}".format(
        pd.read_csv(RUNS / n / "sig_clusters.txt", sep="\t")["cluster_X"].str.count(";").add(1).iloc[0],
        pd.read_csv(RUNS / n / "sig_clusters.txt", sep="\t")["cluster_Y"].str.count(";").add(1).iloc[0],
    )
    for n in RUN_CONFIGS
])
display(summary)

Read that table against the hallagrams below.

**Signed distance** (`--sim2dist_disable_abs`) is the one worth understanding. By default HAllA
clusters on `1 - |similarity|`, so a strongly *anti*-correlated pair counts as similar and lands
in the same block. Disable the absolute value and clustering uses `1 - similarity`, which pushes
anti-correlated features apart. The block *shapes* change even though the underlying correlations
have not.

**NMI** measures shared information after discretisation, so it finds non-monotone relationships
that Spearman misses.

In [ ]:
for name in ["pearson_complete_signed", "nmi_average"]:
    fig = render(name, block_num=10,
                 y_barplot={"values": prev_map, "label": "Prevalence %", "vmax": 100})
    fig.savefig(FIGS / f"hallagram_{name}.png", dpi=200, bbox_inches="tight")
    print(RUN_CONFIGS[name]["label"])
    display(fig)
    plt.close(fig)

## 9. Where to go next

Things worth trying on this data:

- Other metrics: `-m dcor`, `-m mi`, `-m xicor`. `xicor` and `dcor` both pick up non-monotone
  structure, and unlike `nmi` they do not need discretisation.
- `--alla` skips the hierarchy entirely and tests every pair independently. Compare its output
  with the block version to see what the clustering is buying you.
- Subset the samples. `data/sample_metadata.txt` has the study group, so you can run controls and
  adenomas separately and compare the blocks. Watch the sample count: HAllA's power drops quickly.
- Swap the strips. `taxa_metadata.txt` also carries `Family`, `Order` and `mean_log10_abundance`,
  so `y_metadata_columns=["Phylum", "mean_log10_abundance"]` gives you a numeric strip on the
  taxa axis too.
